In [37]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import openpyxl
import re 

import warnings
warnings.filterwarnings("ignore")

In [38]:
start_time = time.time()

In [39]:
bd.projects.set_current('nitrous_oxide_production_monte_carlo')

In [40]:
sorted(list(bd.databases))

['ecoinvent-3.10-biosphere', 'ecoinvent-3.10-cutoff', 'nitrous-oxide-ei310']

In [41]:
methods = [('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]
len(methods)

1

In [42]:
methods

[('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]

In [43]:
db = bd.Database('nitrous-oxide-ei310')
acts = [act for act in db if 'hydrogen peroxide production' in act['name'] or 'nitrous oxide production' in act['name'] or 'phenol production' in act['name']]
len(acts)

14

In [44]:
eidb = bd.Database('ecoinvent-3.10-cutoff')
act = [a for a in eidb if 'phenol production, cumene oxidation' in a['name'] and 'phenol' in a['reference product'] and 'RoW' in a['location']][0]
acts = acts + [act]
len(acts)

15

In [45]:
mc_df = pd.DataFrame()

In [46]:
demands = [{act:1} for act in acts]
all_demands = {k: 1 for demand in demands for k in demand}
lca = bc.LCA(demand=all_demands, method=methods[0], use_distributions=True)
lca.lci()

C_matrices = {}
for method in methods:
    lca.switch_method(method)
    C_matrices[method] = lca.characterization_matrix.copy()
    
results = {
    act['name']: [] for act in acts
}

for _ in range(1000):
    next(lca)
    for index, demand in enumerate(demands):
        lca.lci({key.id: value for key, value in demand.items()})
        for method in methods:
            name = str(list(demand.keys())[0])
            name = name.replace("dict_keys([", "").replace("])", "").replace("'", "")
            name = name[:name.find(' (')]
            results[name].append(
                (C_matrices[method] * lca.inventory).sum()
            )

for key, value in results.items():
    mc_df[key] = pd.Series(value)

In [47]:
mc_df

,"nitrous oxide production, AON 90 cryogenic, green ammonia","phenol production, hydrogen peroxide, fossil","nitrous oxide production, AON 90 cryogenic, fossil ammonia","phenol production, hydrogen peroxide, green","nitrous oxide production, AON 90 membrane, fossil ammonia","phenol production, nitrous oxide, fossil","phenol production, nitrous oxide, green","nitrous oxide production, BAU, green ammonia","hydrogen peroxide production, AO, fossil hydrogen","nitrous oxide production, BAU, fossil ammonia","nitrous oxide production, AON 90 membrane, green ammonia","hydrogen peroxide production, AO, green hydrogen","nitrous oxide production, AON 100, green ammonia","nitrous oxide production, AON 100, fossil ammonia","phenol production, cumene oxidation"
0,2.638627,3.966675,4.505712,3.667135,2.895683,5.455443,3.928766,1.578872,1.223567,3.731582,0.921664,0.611803,0.941519,2.697005,6.016539
1,1.788003,3.495901,4.541709,2.873688,2.914286,4.656668,3.553568,1.061760,1.222305,3.665922,0.671805,0.446346,0.734866,2.878709,6.541391
2,2.260944,4.324551,4.597227,3.651359,3.754950,5.676648,4.477752,1.298360,1.235330,3.879565,0.978368,0.545938,0.954995,3.492834,6.033594
3,2.002751,3.114647,4.619090,3.164614,3.002964,4.506024,3.641026,1.238669,1.253017,3.348206,0.967418,0.519383,0.821929,2.762699,5.695223
4,2.059778,3.874525,4.763477,2.950368,3.605703,5.265887,3.566049,1.247574,1.289598,3.843592,1.054965,0.521802,0.621467,2.742113,5.893087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2.316270,3.250794,4.466355,2.679966,2.771842,4.194452,3.139493,1.314110,1.319937,3.482209,1.195783,0.617937,0.985985,3.066773,5.316797
996,2.522890,3.793813,4.600817,3.268037,3.815677,4.800685,3.304280,1.746996,1.310475,4.728694,1.232250,0.629733,1.020569,3.446365,5.454509
997,2.142637,4.208691,4.669112,4.344087,3.274870,5.667284,4.264220,1.394196,1.173213,4.369925,1.077280,0.625329,0.819181,3.723260,6.375946
998,2.453084,3.072912,4.990715,3.263265,3.352143,4.139383,3.199473,1.356682,1.308439,3.363279,1.089065,0.749134,0.931837,2.945570,5.383345


In [48]:
mc_df.to_excel(os.path.join('..', 'results', 'monte_carlo.xlsx'))